# Week 3: Post-Class Exercise (ADVANCED) - Handwritten Digit Classification (SOLUTIONS)

**Dataset:** Handwritten Digits (8x8 pixel images)  
**Complete solutions with all TODOs filled in**

---

## Part 1: Setup and Data Loading

In [ ]:
# SOLUTION: Import all libraries

import numpy as np
import pandas as pd

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score
)

import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

print("✅ All imports successful!")

In [ ]:
# SOLUTION: Load Digits dataset

digits = load_digits()

X = digits.data
y = digits.target
images = digits.images

print("✅ Data loaded!")
print(f"Shape: X={X.shape}, y={y.shape}")
print(f"Images shape: {images.shape}")
print(f"\nFeature range: {X.min():.0f} to {X.max():.0f}")
print(f"Classes: {np.unique(y)}")

In [ ]:
# Visualize sample digits

fig, axes = plt.subplots(2, 10, figsize=(12, 3))
axes = axes.ravel()

for i in range(20):
    axes[i].imshow(images[i], cmap='gray')
    axes[i].set_title(f'Label: {y[i]}')
    axes[i].axis('off')

plt.suptitle('Sample Handwritten Digits (8x8 pixels)', y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# SOLUTION: Check class distribution

print("Digit distribution:")
digit_counts = pd.Series(y).value_counts().sort_index()
print(digit_counts)

print(f"\nClass balance:")
for digit in range(10):
    count = (y == digit).sum()
    print(f"  Digit {digit}: {count} ({count/len(y)*100:.1f}%)")

**SOLUTION Question 1.1:** Is this dataset balanced?

*Answer:* **Yes, nearly balanced** - each digit appears 174-183 times (~10% each, very close to perfect 10%). This means accuracy is a reliable metric and we don't need special handling for class imbalance.

In [ ]:
# Visualize class distribution

plt.figure(figsize=(10, 4))
plt.bar(range(10), [sum(y == i) for i in range(10)], color='steelblue')
plt.xlabel('Digit')
plt.ylabel('Count')
plt.title('Distribution of Digits in Dataset')
plt.xticks(range(10))
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# SOLUTION: Feature statistics

df = pd.DataFrame(X, columns=[f'pixel_{i}' for i in range(64)])
df['digit'] = y

print("Feature statistics (first 10 pixels):")
print(df.iloc[:, :10].describe())

**SOLUTION Question 1.2:** Which pixels have most information?

*Answer:* **Center pixels (columns 3-6, rows 3-6)** likely have highest variance because:
- Edge and corner pixels are mostly white (background)
- Center pixels contain the actual digit strokes
- Higher variance = more discriminative power

## Part 2: Data Preparation

In [ ]:
# SOLUTION: Train/test split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("✅ Data split complete!")
print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

In [ ]:
# SOLUTION: Feature scaling

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("✅ Features scaled!")
print(f"\nTraining data (first pixel):")
print(f"  Mean: {X_train_scaled[:, 0].mean():.4f}")
print(f"  Std:  {X_train_scaled[:, 0].std():.4f}")

## Part 3: Model Training (SOLUTIONS)

In [ ]:
# SOLUTION: Train Decision Tree with max_depth=15
# With 64 features, we can go deeper than with 4 features (Iris)

dt_model = DecisionTreeClassifier(
    max_depth=15,  # Higher than Iris (5) because more features
    random_state=42
)

dt_model.fit(X_train_scaled, y_train)
dt_pred = dt_model.predict(X_test_scaled)

print("✅ Decision Tree trained!")
print(f"First 10 predictions: {dt_pred[:10]}")

In [ ]:
# SOLUTION: Train Random Forest

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,  # Let trees grow fully (averaging prevents overfitting)
    random_state=42
)

rf_model.fit(X_train_scaled, y_train)
rf_pred = rf_model.predict(X_test_scaled)

print("✅ Random Forest trained!")
print(f"First 10 predictions: {rf_pred[:10]}")

In [ ]:
# SOLUTION: Train XGBoost

xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    eval_metric='mlogloss'
)

xgb_model.fit(X_train_scaled, y_train)
xgb_pred = xgb_model.predict(X_test_scaled)

print("✅ XGBoost trained!")
print(f"First 10 predictions: {xgb_pred[:10]}")

## Part 4: Model Evaluation (SOLUTIONS)

In [ ]:
# SOLUTION: Calculate metrics for Decision Tree

dt_accuracy = accuracy_score(y_test, dt_pred)
dt_precision = precision_score(y_test, dt_pred, average='weighted')
dt_recall = recall_score(y_test, dt_pred, average='weighted')
dt_f1 = f1_score(y_test, dt_pred, average='weighted')

print("Decision Tree Metrics:")
print(f"Accuracy:  {dt_accuracy:.3f}")
print(f"Precision: {dt_precision:.3f}")
print(f"Recall:    {dt_recall:.3f}")
print(f"F1 Score:  {dt_f1:.3f}")

In [ ]:
# SOLUTION: Calculate metrics for Random Forest

rf_accuracy = accuracy_score(y_test, rf_pred)
rf_precision = precision_score(y_test, rf_pred, average='weighted')
rf_recall = recall_score(y_test, rf_pred, average='weighted')
rf_f1 = f1_score(y_test, rf_pred, average='weighted')

print("Random Forest Metrics:")
print(f"Accuracy:  {rf_accuracy:.3f}")
print(f"Precision: {rf_precision:.3f}")
print(f"Recall:    {rf_recall:.3f}")
print(f"F1 Score:  {rf_f1:.3f}")

In [ ]:
# SOLUTION: Calculate metrics for XGBoost

xgb_accuracy = accuracy_score(y_test, xgb_pred)
xgb_precision = precision_score(y_test, xgb_pred, average='weighted')
xgb_recall = recall_score(y_test, xgb_pred, average='weighted')
xgb_f1 = f1_score(y_test, xgb_pred, average='weighted')

print("XGBoost Metrics:")
print(f"Accuracy:  {xgb_accuracy:.3f}")
print(f"Precision: {xgb_precision:.3f}")
print(f"Recall:    {xgb_recall:.3f}")
print(f"F1 Score:  {xgb_f1:.3f}")

In [ ]:
# SOLUTION: Generate confusion matrices

cm_dt = confusion_matrix(y_test, dt_pred)
cm_rf = confusion_matrix(y_test, rf_pred)
cm_xgb = confusion_matrix(y_test, xgb_pred)

print("✅ Confusion matrices generated!")
print(f"Decision Tree confusion matrix shape: {cm_dt.shape}")

In [ ]:
# Visualize 10x10 confusion matrices

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.heatmap(cm_dt, annot=True, fmt='d', cmap='Blues', ax=axes[0], cbar_kws={'label': 'Count'})
axes[0].set_title('Decision Tree Confusion Matrix')
axes[0].set_ylabel('Actual Digit')
axes[0].set_xlabel('Predicted Digit')

sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Greens', ax=axes[1], cbar_kws={'label': 'Count'})
axes[1].set_title('Random Forest Confusion Matrix')
axes[1].set_ylabel('Actual Digit')
axes[1].set_xlabel('Predicted Digit')

sns.heatmap(cm_xgb, annot=True, fmt='d', cmap='Oranges', ax=axes[2], cbar_kws={'label': 'Count'})
axes[2].set_title('XGBoost Confusion Matrix')
axes[2].set_ylabel('Actual Digit')
axes[2].set_xlabel('Predicted Digit')

plt.tight_layout()
plt.show()

**SOLUTION Question 4.1:**

a) **Which digit is easiest to classify?**

*Answer:* Typically **digit 0 or 1** - these have very distinctive shapes (circle and vertical line) that are hard to confuse with other digits.

b) **Which two digits are most commonly confused?**

*Answer:* Common confusions (check your confusion matrix):
- **4 and 9** - Similar diagonal strokes
- **3 and 8** - Both have curved elements
- **7 and 9** - Similar top structure

c) **Why might these digits be confused?**

*Answer:* At low resolution (8x8 pixels), subtle differences disappear:
- Handwriting variations make similar shapes overlap
- 8x8 resolution loses fine details
- Diagonal strokes pixelate similarly

In [ ]:
# SOLUTION: Classification report

print("="*70)
print("CLASSIFICATION REPORT (Best Model - typically RF or XGB)")
print("="*70)
print(classification_report(y_test, xgb_pred, target_names=[str(i) for i in range(10)]))

**SOLUTION Question 4.2:**

a) **Which digit has the lowest F1 score?**

*Answer:* Typically **digit 8** or **digit 9** (check your report).

b) **Why might this digit be harder?**

*Answer:* 
- **Digit 8:** Complex shape with two loops, can be confused with 3, 6, or 0
- **Digit 9:** Similar to 4 and 7 depending on handwriting style
- At 8x8 resolution, intricate shapes lose detail

## Part 5: Overfitting Analysis (SOLUTIONS)

In [ ]:
# SOLUTION: Calculate training accuracy

dt_train_acc = dt_model.score(X_train_scaled, y_train)
rf_train_acc = rf_model.score(X_train_scaled, y_train)
xgb_train_acc = xgb_model.score(X_train_scaled, y_train)

# Compare train vs test
print("="*80)
print("OVERFITTING ANALYSIS: Train vs Test Accuracy")
print("="*80)
print(f"{'Model':<20} {'Train Acc':<12} {'Test Acc':<12} {'Gap':<12} {'Status':<15}")
print("-"*80)

def overfitting_status(gap):
    if gap < 0.02:
        return "✅ Good"
    elif gap < 0.05:
        return "⚠️ Mild overfit"
    else:
        return "❌ Overfitting"

dt_gap = dt_train_acc - dt_accuracy
rf_gap = rf_train_acc - rf_accuracy
xgb_gap = xgb_train_acc - xgb_accuracy

print(f"{'Decision Tree':<20} {dt_train_acc:<12.3f} {dt_accuracy:<12.3f} {dt_gap:<12.3f} {overfitting_status(dt_gap):<15}")
print(f"{'Random Forest':<20} {rf_train_acc:<12.3f} {rf_accuracy:<12.3f} {rf_gap:<12.3f} {overfitting_status(rf_gap):<15}")
print(f"{'XGBoost':<20} {xgb_train_acc:<12.3f} {xgb_accuracy:<12.3f} {xgb_gap:<12.3f} {overfitting_status(xgb_gap):<15}")
print("="*80)

**SOLUTION Question 5.1:**

a) **Which model has the largest train/test gap?**

*Answer:* Typically **Random Forest** (often ~100% train, ~97% test = 3% gap) or **Decision Tree** if max_depth is too high.

b) **What does a large gap (>5%) indicate?**

*Answer:* **Overfitting** - the model has:
- Memorized training data patterns (including noise)
- Failed to learn generalizable rules
- Becomes less reliable on new data
- Needs regularization (lower max_depth, more pruning)

c) **How to reduce overfitting?**

*Answer:* 
- **Decision Tree:** Reduce max_depth (from 15 to 10), increase min_samples_split
- **Random Forest:** Usually robust, but could reduce max_depth per tree
- **XGBoost:** Reduce learning_rate, increase regularization (reg_alpha, reg_lambda)

## Part 6: Feature Importance (SOLUTIONS)

In [ ]:
# SOLUTION: Extract feature importance

dt_importance = dt_model.feature_importances_
rf_importance = rf_model.feature_importances_
xgb_importance = xgb_model.feature_importances_

print("✅ Feature importance extracted!")

In [ ]:
# Visualize as 8x8 heatmaps

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].imshow(dt_importance.reshape(8, 8), cmap='hot', interpolation='nearest')
axes[0].set_title('Decision Tree\nFeature Importance')
axes[0].axis('off')

axes[1].imshow(rf_importance.reshape(8, 8), cmap='hot', interpolation='nearest')
axes[1].set_title('Random Forest\nFeature Importance')
axes[1].axis('off')

im = axes[2].imshow(xgb_importance.reshape(8, 8), cmap='hot', interpolation='nearest')
axes[2].set_title('XGBoost\nFeature Importance')
axes[2].axis('off')

fig.colorbar(im, ax=axes, orientation='horizontal', fraction=0.05, pad=0.1, label='Importance')

plt.suptitle('Which Pixels Matter Most? (Brighter = More Important)', y=1.05)
plt.tight_layout()
plt.show()

**SOLUTION Question 6.1:**

a) **Which region has highest importance?**

*Answer:* **Center region (rows 2-5, columns 2-5)** - the middle pixels of the 8x8 image.

b) **Why does this make sense?**

*Answer:* 
- Handwritten digits are centered in the image
- Edge and corner pixels are mostly background (white)
- Center pixels contain the actual digit strokes
- Most discriminative information is where ink appears

c) **Do all models agree?**

*Answer:* **Mostly yes** - all three show center pixels as most important, but:
- Decision Tree might focus on specific pixels (single best splits)
- Random Forest averages across many trees (more distributed importance)
- XGBoost finds sequential patterns (might highlight different regions)

The general pattern (center > edges) is consistent across all models.

## Part 7: Algorithm Comparison (SOLUTIONS)

In [ ]:
# SOLUTION: Comparison table

comparison_df = pd.DataFrame({
    'Model': ['Decision Tree', 'Random Forest', 'XGBoost'],
    'Accuracy': [dt_accuracy, rf_accuracy, xgb_accuracy],
    'F1 Score': [dt_f1, rf_f1, xgb_f1],
    'Train Acc': [dt_train_acc, rf_train_acc, xgb_train_acc],
    'Overfit Gap': [dt_gap, rf_gap, xgb_gap]
})

print('='*70)
print('MODEL COMPARISON: Handwritten Digit Classification')
print('='*70)
print(comparison_df.to_string(index=False))
print('='*70)

best_accuracy = comparison_df['Accuracy'].idxmax()
best_f1 = comparison_df['F1 Score'].idxmax()
least_overfit = comparison_df['Overfit Gap'].idxmin()

print(f"\n🏆 Best Accuracy: {comparison_df.loc[best_accuracy, 'Model']}")
print(f"🏆 Best F1 Score: {comparison_df.loc[best_f1, 'Model']}")
print(f"🏆 Least Overfitting: {comparison_df.loc[least_overfit, 'Model']}")

## Part 8: Critical Thinking (SOLUTIONS)

**SOLUTION Question 8.1: Production Deployment**

a) **Based on test accuracy alone?**

*Answer:* Typically **Random Forest or XGBoost** (~97-98% accuracy).

b) **Considering speed for thousands of envelopes per minute?**

*Answer:* **Decision Tree** because:

| Model | Prediction Speed | Test Accuracy | Production Viability |
|-------|-----------------|---------------|---------------------|
| Decision Tree | ⚡⚡⚡ Fastest (single tree) | ~85-90% | ✅ Good enough + fast |
| Random Forest | ⚡ Slow (100 trees) | ~97% | ⚠️ Accuracy great but too slow |
| XGBoost | ⚡ Slow (100 sequential) | ~97% | ⚠️ Accuracy great but too slow |

**For postal service:** Decision Tree with max_depth=10-15 achieves ~85-90% accuracy at 100x faster speed.

**Alternative:** Use Random Forest but reduce trees (n_estimators=10) for speed/accuracy tradeoff.

c) **Cost calculation:**

*Example with typical results:*
- **XGBoost:** 97% accuracy = 3% error
  - Daily errors: 1,000,000 × 0.03 = 30,000 mistakes
  - Daily cost: 30,000 × $5 = **$150,000**

- **Decision Tree:** 87% accuracy = 13% error
  - Daily errors: 1,000,000 × 0.13 = 130,000 mistakes
  - Daily cost: 130,000 × $5 = **$650,000**

**Difference:** $500,000/day extra cost with Decision Tree!

**Conclusion:** At this scale, **accuracy matters more than speed** - worth the computational cost.

d) **Final recommendation:**

*Answer:* **Random Forest with n_estimators=50** because:
- Balances accuracy (~96%) and speed (2x faster than n=100)
- More robust than XGBoost (easier to maintain)
- Can parallelize across multiple CPUs
- Saves ~$400K/day vs Decision Tree

**Or** invest in faster hardware and use full Random Forest (100 trees) for maximum accuracy.

**SOLUTION Question 8.2: Comparison with Iris**

a) **How does Digits differ from Iris in complexity?**

*Answer:*

| Aspect | Iris | Digits | Impact |
|--------|------|--------|--------|
| **Classes** | 3 | 10 | 3.3x more ways to make mistakes |
| **Features** | 4 | 64 | 16x higher dimensionality |
| **Samples** | 150 | 1,797 | 12x more data (good!) |
| **Complexity** | Low | High | More overfitting risk |

**Result:** Digits is significantly harder:
- Confusion matrix: 9 cells (3×3) → 100 cells (10×10)
- Feature space: 4D → 64D (curse of dimensionality)
- More opportunities for overfitting

b) **More overfitting in Digits than Iris?**

*Answer:* **Yes, significantly more overfitting in Digits:**

**Iris results (typical):**
- Decision Tree: Train 98%, Test 95% (gap: 3%)
- Random Forest: Train 100%, Test 97% (gap: 3%)

**Digits results (typical):**
- Decision Tree: Train 95%, Test 87% (gap: 8%) ← **More overfitting!**
- Random Forest: Train 100%, Test 97% (gap: 3%)

**Why:** 
- 64 features vs 4 → model has more dimensions to memorize noise
- Pixel data has natural noise (handwriting variations)
- Iris features (petal measurements) are cleaner

c) **Which dataset better demonstrates ensemble power?**

*Answer:* **Digits** demonstrates ensemble methods better:

**On Iris:**
- All three models get ~95-100% accuracy
- Hard to see performance differences
- Too easy for modern algorithms

**On Digits:**
- Clear performance gap: DT (~85-90%) < RF (~97%) ≈ XGB (~97%)
- **Ensemble advantage is obvious** (~10% accuracy improvement)
- Overfitting differences are visible
- Real-world complexity (speed vs accuracy tradeoff matters)

**Digits is better for learning** because it shows WHY we use ensemble methods!

## Part 9: Error Analysis (BONUS)

In [ ]:
# BONUS: Visualize misclassifications

best_model_name = comparison_df.loc[comparison_df['F1 Score'].idxmax(), 'Model']
if best_model_name == 'Decision Tree':
    best_pred = dt_pred
elif best_model_name == 'Random Forest':
    best_pred = rf_pred
else:
    best_pred = xgb_pred

misclassified_mask = best_pred != y_test
misclassified_indices = np.where(misclassified_mask)[0]

print(f"Model: {best_model_name}")
print(f"Total misclassified: {sum(misclassified_mask)} out of {len(y_test)}")

if len(misclassified_indices) > 0:
    fig, axes = plt.subplots(2, 5, figsize=(12, 5))
    axes = axes.ravel()
    
    test_images = X_test.reshape(-1, 8, 8)
    
    for i, idx in enumerate(misclassified_indices[:10]):
        axes[i].imshow(test_images[idx], cmap='gray')
        axes[i].set_title(f'True: {y_test[idx]}\nPred: {best_pred[idx]}', color='red')
        axes[i].axis('off')
    
    plt.suptitle(f'First 10 Misclassifications ({best_model_name})', color='red', fontsize=14)
    plt.tight_layout()
    plt.show()
    
    print("\nLook at these mistakes - can YOU see why the model was confused?")
    print("At 8x8 resolution, even humans might struggle with some of these!")
else:
    print("No misclassifications! Perfect model!")

---

## Key Takeaways (SOLUTIONS)

**What we learned from Digits vs Iris:**

1. **Complexity matters:**
   - 10 classes harder than 3 classes
   - 64 features harder than 4 features
   - More features = more overfitting risk

2. **Ensemble methods shine on complex data:**
   - Iris: All models ~95-100% (too easy)
   - Digits: DT ~87%, RF/XGB ~97% (clear winner)
   - **~10% accuracy gain justifies ensemble complexity**

3. **Overfitting is real:**
   - Train/test gap much larger on Digits
   - max_depth matters more with 64 features
   - Random Forest averaging prevents overfitting

4. **Feature importance visualization:**
   - Center pixels matter most
   - Edge pixels are background noise
   - Can reshape 64 features → 8×8 image

5. **Real-world tradeoffs:**
   - Speed vs accuracy (DT fast, RF/XGB slow)
   - Cost of errors ($500K/day difference!)
   - Production deployment considerations

**Recommendation for students:**
- **Start with Iris** to learn multi-class basics
- **Progress to Digits** to see real-world complexity
- Digits better demonstrates WHY ensemble methods exist!

---

*Week 3 Post-Class Solutions (Advanced - Digits) v1.0 | February 2026*